# Match Citations to PDF Files
## Reads references.txt and finds corresponding PDF files in Papers/ folder

In [1]:
# Import libraries
import os
import re
import glob
from difflib import SequenceMatcher
import shutil

print("✓ Imports successful")

✓ Imports successful


In [2]:
# Configuration
REFERENCES_FILE = './references.txt'
PAPERS_FOLDER = './Papers'
OUTPUT_FOLDER = './cited_papers'  # Optional: where to copy cited papers

print(f"References file: {REFERENCES_FILE}")
print(f"Papers folder: {PAPERS_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")

References file: ./references.txt
Papers folder: ./Papers
Output folder: ./cited_papers


In [3]:
# Load references.txt
print("\nLoading references...")

with open(REFERENCES_FILE, 'r', encoding='utf-8') as f:
    content = f.read()

# Parse references - format: [1] Author - papers/filename.pdf
reference_pattern = r'\[(\d+)\]\s+(.+?)\s+-\s+papers/(.+\.pdf)'
references = []

for match in re.finditer(reference_pattern, content):
    cite_num = int(match.group(1))
    author = match.group(2).strip()
    filename = match.group(3).strip()
    
    references.append({
        'cite_num': cite_num,
        'author': author,
        'filename': filename
    })

print(f"✓ Found {len(references)} references")
print(f"\nFirst 5 references:")
for ref in references[:5]:
    print(f"  [{ref['cite_num']}] {ref['filename'][:60]}")


Loading references...
✓ Found 48 references

First 5 references:
  [1] INDOOR TESTBED FOR VECTOR FIELD MULTIROBOT ADAPTIVE NAVIGATI
  [2] Vector Field Based Collision Avoidance2207.01747.pdf
  [3] [8] Motion planning_and collision_avoidance_using_navigation
  [4] Initial_Study_of_Multirobot_Adaptive_Navigation_for_Explorin
  [5] [108] Experimental Implementation and Verification of Scalar


In [4]:
# Scan Papers folder for all PDFs
print(f"\nScanning {PAPERS_FOLDER} for PDF files...")

pdf_files = glob.glob(f"{PAPERS_FOLDER}/*.pdf")
pdf_basenames = [os.path.basename(f) for f in pdf_files]

print(f"✓ Found {len(pdf_files)} PDF files in Papers/ folder")


Scanning ./Papers for PDF files...
✓ Found 57 PDF files in Papers/ folder


In [5]:
# Match references to actual files
print("\n" + "="*70)
print("MATCHING CITATIONS TO FILES")
print("="*70)

def similarity(a, b):
    """Calculate similarity ratio between two strings"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

matches = []
exact_matches = 0
fuzzy_matches = 0
no_matches = 0

for ref in references:
    ref_filename = ref['filename']
    
    # Try exact match first
    if ref_filename in pdf_basenames:
        actual_file = os.path.join(PAPERS_FOLDER, ref_filename)
        matches.append({
            'cite_num': ref['cite_num'],
            'reference_filename': ref_filename,
            'actual_file': actual_file,
            'match_type': 'exact',
            'similarity': 1.0
        })
        exact_matches += 1
    else:
        # Try fuzzy matching
        best_match = None
        best_score = 0.0
        
        for pdf_basename in pdf_basenames:
            score = similarity(ref_filename, pdf_basename)
            if score > best_score:
                best_score = score
                best_match = pdf_basename
        
        if best_score > 0.7:  # Threshold for fuzzy match
            actual_file = os.path.join(PAPERS_FOLDER, best_match)
            matches.append({
                'cite_num': ref['cite_num'],
                'reference_filename': ref_filename,
                'actual_file': actual_file,
                'match_type': 'fuzzy',
                'similarity': best_score
            })
            fuzzy_matches += 1
        else:
            matches.append({
                'cite_num': ref['cite_num'],
                'reference_filename': ref_filename,
                'actual_file': None,
                'match_type': 'not_found',
                'similarity': best_score
            })
            no_matches += 1

print(f"\nMatching results:")
print(f"  ✓ Exact matches: {exact_matches}")
print(f"  ≈ Fuzzy matches: {fuzzy_matches}")
print(f"  ✗ Not found: {no_matches}")
print(f"  Total: {len(matches)}")


MATCHING CITATIONS TO FILES

Matching results:
  ✓ Exact matches: 47
  ≈ Fuzzy matches: 1
  ✗ Not found: 0
  Total: 48


In [6]:
# Display all matches
print("\n" + "="*70)
print("DETAILED MATCH RESULTS")
print("="*70)

print("\n✓ EXACT MATCHES:\n")
for match in matches:
    if match['match_type'] == 'exact':
        print(f"  [{match['cite_num']:<3}] {match['reference_filename'][:70]}")

if fuzzy_matches > 0:
    print("\n≈ FUZZY MATCHES (verify these!):\n")
    for match in matches:
        if match['match_type'] == 'fuzzy':
            print(f"  [{match['cite_num']:<3}] Similarity: {match['similarity']:.2f}")
            print(f"        Reference: {match['reference_filename']}")
            print(f"        Found:     {os.path.basename(match['actual_file'])}")
            print()

if no_matches > 0:
    print("\n✗ NOT FOUND (these PDFs are missing!):\n")
    for match in matches:
        if match['match_type'] == 'not_found':
            print(f"  [{match['cite_num']:<3}] {match['reference_filename'][:70]}")


DETAILED MATCH RESULTS

✓ EXACT MATCHES:

  [1  ] INDOOR TESTBED FOR VECTOR FIELD MULTIROBOT ADAPTIVE NAVIGATION.pdf
  [2  ] Vector Field Based Collision Avoidance2207.01747.pdf
  [3  ] [8] Motion planning_and collision_avoidance_using_navigation_vector_fi
  [4  ] Initial_Study_of_Multirobot_Adaptive_Navigation_for_Exploring_Environm
  [5  ] [108] Experimental Implementation and Verification of Scalar Field Rid
  [6  ] [14] Gradient-based_Adaptive_Navigation_of_Multiple_Diving_Autonomous_
  [7  ] [9] Navigation_of_Scalar_Fronts_With_Multirobot_Clusters_in_Simulation
  [9  ] [101] obstacle avoidance using complex vector fields.pdf
  [10 ] [102] Structured Isosurface Mapping of 3D Scalar Fields with Mobile Se
  [11 ] Distributed_Multi-Robot_Active-Sensing_of_a_Diffusive_Source.pdf
  [12 ] Unifying_Control_Architecture_for_Reactive_Particle_Swarms.pdf
  [13 ] [20] garg-panagou-2019-finite-time-estimation-and-control-for-multi-ai
  [14 ] [106] Obstacle Avoidance Policies for Cluster Space

In [7]:
# Optional: Copy cited papers to a separate folder
print("\n" + "="*70)
print("COPY CITED PAPERS TO SEPARATE FOLDER?")
print("="*70)

copy_files = input("\nCopy cited papers to ./cited_papers/? (y/n): ").strip().lower()

if copy_files == 'y':
    # Create output folder
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    copied = 0
    skipped = 0
    
    for match in matches:
        if match['actual_file'] and os.path.exists(match['actual_file']):
            # Create new filename with citation number prefix
            cite_num = match['cite_num']
            original_name = os.path.basename(match['actual_file'])
            new_name = f"[{cite_num:03d}] {original_name}"
            
            dest_path = os.path.join(OUTPUT_FOLDER, new_name)
            shutil.copy2(match['actual_file'], dest_path)
            
            copied += 1
        else:
            skipped += 1
    
    print(f"\n✓ Copied {copied} papers to {OUTPUT_FOLDER}")
    if skipped > 0:
        print(f"⚠ Skipped {skipped} papers (files not found)")
else:
    print("\nSkipped copying.")


COPY CITED PAPERS TO SEPARATE FOLDER?

✓ Copied 48 papers to ./cited_papers


In [8]:
# Generate summary report
print("\n" + "="*70)
print("SUMMARY REPORT")
print("="*70)

# Create lookup table
print("\nCitation Number → PDF Filename Lookup:\n")
print(f"{'Cite':<6} {'Status':<10} Filename")
print("-"*90)

for match in sorted(matches, key=lambda x: x['cite_num']):
    cite_num = f"[{match['cite_num']}]"
    
    if match['match_type'] == 'exact':
        status = "✓ Found"
        filename = os.path.basename(match['actual_file'])
    elif match['match_type'] == 'fuzzy':
        status = f"≈ {match['similarity']:.0%}"
        filename = os.path.basename(match['actual_file'])
    else:
        status = "✗ Missing"
        filename = match['reference_filename']
    
    print(f"{cite_num:<6} {status:<10} {filename[:70]}")

print("\n" + "="*70)
print(f"Total citations: {len(matches)}")
print(f"Files found: {exact_matches + fuzzy_matches}/{len(matches)}")
print(f"Files missing: {no_matches}/{len(matches)}")

if no_matches == 0:
    print("\n🎉 All cited papers found!")
else:
    print(f"\n⚠ Warning: {no_matches} cited papers are missing from Papers/ folder")

print("="*70)


SUMMARY REPORT

Citation Number → PDF Filename Lookup:

Cite   Status     Filename
------------------------------------------------------------------------------------------
[1]    ✓ Found    INDOOR TESTBED FOR VECTOR FIELD MULTIROBOT ADAPTIVE NAVIGATION.pdf
[2]    ✓ Found    Vector Field Based Collision Avoidance2207.01747.pdf
[3]    ✓ Found    [8] Motion planning_and collision_avoidance_using_navigation_vector_fi
[4]    ✓ Found    Initial_Study_of_Multirobot_Adaptive_Navigation_for_Exploring_Environm
[5]    ✓ Found    [108] Experimental Implementation and Verification of Scalar Field Rid
[6]    ✓ Found    [14] Gradient-based_Adaptive_Navigation_of_Multiple_Diving_Autonomous_
[7]    ✓ Found    [9] Navigation_of_Scalar_Fronts_With_Multirobot_Clusters_in_Simulation
[8]    ≈ 100%      Adaptive Navigation Control Primitives for Multirobot Clusters- Extre
[9]    ✓ Found    [101] obstacle avoidance using complex vector fields.pdf
[10]   ✓ Found    [102] Structured Isosurface Mapping of 3D 

In [9]:
# Export results to CSV (optional)
import csv

csv_file = './citation_to_file_mapping.csv'

with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['cite_num', 'reference_filename', 'actual_file', 'match_type', 'similarity'])
    writer.writeheader()
    writer.writerows(matches)

print(f"\n✓ Exported mapping to {csv_file}")


✓ Exported mapping to ./citation_to_file_mapping.csv
